In [ ]:
import torch
from transformers import RobertaTokenizer, RobertaConfig, RobertaModel, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from torch import nn 
from tqdm import tqdm

In [6]:
vulBERTa = "claudios/VulBERTa-MLP-ReVeal"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(vulBERTa, trust_remote_code=True)

# Model Finetuning:

# Model Evaluation:

In [7]:
prompt_tokens = tokenizer.tokenize("Does this code have a vulnerability?")
# int main() { int c = 2; int arr[] = {1}; printf(\"%d\", arr[c]);}
file = open('testfile.c', 'r')
code = file.read()
print(code)
code_tokens = tokenizer.tokenize(code)
inputs = tokenizer.encode_plus("".join(prompt_tokens), "".join(code_tokens), add_special_tokens=True, padding=True, truncation=True, return_tensors="pt")
classification_model = AutoModelForSequenceClassification.from_pretrained(vulBERTa)
classification_model.to(device)

for input_data in tqdm([inputs], desc="Processing batch", unit="example"):
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)
    classification_model.eval()
    with torch.no_grad():
        outputs = classification_model(input_ids=input_ids, attention_mask=attention_mask)

logits = outputs.logits
predicted_class = torch.argmax(logits, dim=1).item()
if predicted_class == 1:
    print("The code has a vulnerability.")
else:
    print("The code does not have a vulnerability.")

int main() { int iamveryvulnerable = 2; int arr[] = {1}; printf("%d", arr[iamveryvulnerable]);}


Processing batch: 100%|██████████| 1/1 [00:00<00:00,  2.04example/s]

{0: 'LABEL_0', 1: 'LABEL_1'}
The code does not have a vulnerability.
